# RVC 变声器模型训练 - Kaggle版本

训练 taffie 模型（无音高指导，语音用）

**使用前需要：**
1. 新建 Kaggle Dataset，包含你的音频文件（FLAC/WAV）
2. Notebook 右侧 Settings → Accelerator → **GPU T4 x2**
3. 点击 Add Data → 选择你上传的 Dataset

In [ ]:
# @title 1. 查看硬件
!nvidia-smi
!echo "CPU核数: $(nproc)"

In [ ]:
# @title 2. 克隆 RVC 仓库
!git clone --depth=1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git
%cd /kaggle/working/Retrieval-based-Voice-Conversion-WebUI
!mkdir -p assets/hubert assets/rmvpe assets/pretrained assets/pretrained_v2

In [ ]:
# @title 3. 安装 Python 依赖
# Kaggle 环境: Python 3.12, PyTorch 2.10 已预装
# pip 24.x 拒绝 omegaconf 旧 metadata → 先降级 pip 到 24.0

!echo "=== 降级 pip ==="
!python -m pip install pip==24.0 -q

!echo "=== 安装 fairseq ==="
!pip install fairseq -q

!echo "=== 装其他依赖 ==="
!pip install av librosa soundfile scipy scikit-learn -q
!pip install ffmpeg-python praat-parselmouth pyworld -q
!pip install faiss-cpu tensorboardX matplotlib -q
!pip install gradio requests -q

!echo "=== 打 fairseq Python 3.12 dataclass 兼容补丁 ==="
# Python 3.12 不允许 dataclass 字段用 mutable default（如 x: T = T()）
# 必须用 default_factory。但 fairseq 还没有这个修复。
# 解法: monkeypatch dataclasses._get_field 自动转换
fix_code = r'''
import dataclasses

_orig_get_field = dataclasses._get_field

def _safe_get_field(cls, a_name, a_type, a_default, **kw):
    try:
        return _orig_get_field(cls, a_name, a_type, a_default, **kw)
    except ValueError as e:
        if "mutable default" in str(e):
            import dataclasses as dc
            f = dc.field(default_factory=type(a_default))
            f.name = a_name
            f.type = a_type
            return f
        raise

dataclasses._get_field = _safe_get_field

import fairseq, torch
torch.serialization.add_safe_globals([fairseq.data.dictionary.Dictionary])
print("fairseq OK, torch " + torch.__version__)
'''

with open('fix_fairseq.py', 'w') as f:
    f.write(fix_code)

!python fix_fairseq.py && rm fix_fairseq.py

In [ ]:
# @title 4. 下载预训练模型（HuggingFace）
import requests, os

BASE = 'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/'

files = [
    ('hubert_base.pt', 'assets/hubert/hubert_base.pt'),
    ('pretrained_v2/G40k.pth', 'assets/pretrained_v2/G40k.pth'),
    ('pretrained_v2/D40k.pth', 'assets/pretrained_v2/D40k.pth'),
]

for name, path in files:
    url = BASE + name
    print(f'Downloading {name}...', end=' ')
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    with open(path, 'wb') as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
    sz = os.path.getsize(path)
    print(f'{sz/1024/1024:.1f} MB')

In [ ]:
# @title 5. 打兼容补丁（fairseq dataclass + matplotlib + import hooks）
# 1. 创建修复模块，所有脚本共享
import_code = r'''
import dataclasses

_orig_get_field = dataclasses._get_field

def _safe_get_field(cls, a_name, a_type, a_default, **kw):
    try:
        return _orig_get_field(cls, a_name, a_type, a_default, **kw)
    except ValueError as e:
        if "mutable default" in str(e):
            import dataclasses as dc
            f = dc.field(default_factory=type(a_default))
            f.name = a_name
            f.type = a_type
            return f
        raise

dataclasses._get_field = _safe_get_field

import fairseq, torch
torch.serialization.add_safe_globals([fairseq.data.dictionary.Dictionary])
'''

with open('fix_imports.py', 'w') as f:
    f.write(import_code)

# 2. 让 extract_feature_print.py 自动先加载 fix_imports
with open('infer/modules/train/extract_feature_print.py', 'r') as f:
    code = f.read()
code = 'import fix_imports\n' + code
with open('infer/modules/train/extract_feature_print.py', 'w') as f:
    f.write(code)

# 3. 修 matplotlib tostring_rgb bug
with open('infer/lib/train/utils.py', 'r') as f:
    code = f.read()
code = code.replace('tostring_rgb()', 'tostring_argb()')
code = code.replace(
    '.reshape(fig.canvas.get_width_height()[::-1] + (3,))',
    '.reshape(fig.canvas.get_width_height()[::-1] + (4,))[:,:,1:]'
)
with open('infer/lib/train/utils.py', 'w') as f:
    f.write(code)

# 4. 验证
import fix_imports
from fix_imports import fairseq, torch
print('OK: fairseq ' + fairseq.__version__ + ', torch ' + torch.__version__)

In [ ]:
# @title 6. 解压训练数据
# 在右侧 Add Data 中添加你的 Dataset "taffie"

import os, glob, shutil

DATASET_PATH = '/kaggle/input/taffie'

!echo "=== Dataset 目录结构 ==="
!find {DATASET_PATH} -type f -maxdepth 3 2>/dev/null | head -20 || ls -laR {DATASET_PATH} 2>/dev/null | head -30

# 递归查找所有音频文件（支持子目录）
os.makedirs('train', exist_ok=True)

extensions = ('.wav', '.flac', '.mp3', '.m4a', '.ogg')
audio_files = []
for root, dirs, files in os.walk(DATASET_PATH):
    for f in files:
        if f.lower().endswith(extensions):
            shutil.copy2(os.path.join(root, f), 'train/')
            audio_files.append(f)

print(f'Copied {len(audio_files)} audio files to train/')

In [ ]:
# @title 7. 数据预处理（切片+重采样）
EXP_NAME = 'taffie'
SR = '40000'
N_P = '4'
TRAIN_DIR = '/kaggle/working/Retrieval-based-Voice-Conversion-WebUI/train'
EXP_DIR = f'/kaggle/working/Retrieval-based-Voice-Conversion-WebUI/logs/{EXP_NAME}'

!python infer/modules/train/preprocess.py "{TRAIN_DIR}" {SR} {N_P} "{EXP_DIR}" False 3.0

In [ ]:
# @title 8. 提取 HuBERT 特征（无音高，只用 GPU）
!python infer/modules/train/extract_feature_print.py "cuda:0" 1 0 0 "{EXP_DIR}" v2

In [ ]:
# @title 9. 生成训练文件列表
import os

gt_dir = os.path.join(EXP_DIR, '0_gt_wavs')
f0_dir = os.path.join(EXP_DIR, '2a_f0')
f0nsf_dir = os.path.join(EXP_DIR, '2b-f0nsf')
feat_dir = os.path.join(EXP_DIR, '3_feature768')

lines = []
for fname in sorted(os.listdir(feat_dir)):
    if not fname.endswith('.npy'):
        continue
    base = fname.replace('.npy', '')
    feat = os.path.join(feat_dir, fname)
    gt = os.path.join(gt_dir, base + '.wav')
    f0 = os.path.join(f0_dir, base + '.wav.npy') if os.path.exists(os.path.join(f0_dir, base + '.wav.npy')) else 'None'
    f0nsf = os.path.join(f0nsf_dir, base + '.wav.npy') if os.path.exists(os.path.join(f0nsf_dir, base + '.wav.npy')) else 'None'
    line = f'{gt}|{feat}|{f0}|{f0nsf}|0\n'
    lines.append(line)

with open(os.path.join(EXP_DIR, 'filelist.txt'), 'w') as f:
    f.writelines(lines)
print(f'Generated filelist with {len(lines)} entries')

In [ ]:
# @title 10. 开始训练（无音高，语音用 v2 底模）
import os
os.environ['USE_LIBUV'] = '0'

!python infer/modules/train/train.py \
    -e "{EXP_NAME}" \
    -sr 40k \
    -f0 0 \
    -bs 8 \
    -g 0 \
    -te 30 \
    -se 10 \
    -pg "assets/pretrained_v2/G40k.pth" \
    -pd "assets/pretrained_v2/D40k.pth" \
    -l 1 \
    -c 0 \
    -sw 0 \
    -v v2

In [ ]:
# @title 11. 训练特征索引（推理时需要）
import os, numpy as np, faiss
from sklearn.cluster import MiniBatchKMeans

feature_dir = os.path.join(EXP_DIR, '3_feature768')
listdir_res = sorted(os.listdir(feature_dir))
npys = []
for name in listdir_res:
    phone = np.load(os.path.join(feature_dir, name))
    npys.append(phone)
big_npy = np.concatenate(npys, 0)
big_npy_idx = np.arange(big_npy.shape[0])
np.random.shuffle(big_npy_idx)
big_npy = big_npy[big_npy_idx]

if big_npy.shape[0] > 2e5:
    kmeans = MiniBatchKMeans(n_clusters=10000, batch_size=big_npy.shape[0] // 20, n_init=3, max_iter=300, max_no_improvement=10, verbose=True)
    kmeans.fit(big_npy)
    big_npy = kmeans.cluster_centers_

n_ivf = min(int(big_npy.shape[0] // 20), 65536)
index = faiss.index_factory(big_npy.shape[1], f"IVF{n_ivf},Flat")
index.train(big_npy)
index.add(big_npy)
faiss.write_index(index, os.path.join(EXP_DIR, f'added_{EXP_NAME}.index'))
print(f'Index saved: added_{EXP_NAME}.index ({big_npy.shape[0]} vectors)')

In [ ]:
# @title 12. 复制模型到 /kaggle/working/（可下载）
import glob, shutil
os.makedirs('/kaggle/working/output', exist_ok=True)

# 最新生成器模型
for f in glob.glob(os.path.join(EXP_DIR, '*_2333333.pth')):
    shutil.copy2(f, '/kaggle/working/output/')
# 索引文件
for f in glob.glob(os.path.join(EXP_DIR, '*.index')):
    shutil.copy2(f, '/kaggle/working/output/')

print('Output files:')
!ls -lh /kaggle/working/output/